In [1]:
import json
import pandas as pd
from collections import defaultdict
from tqdm import tqdm
import os
import random

In [2]:
experiment_name = "orca"
base_path = "/mbz/users/liyuan/LLaMA-Factory"

In [ ]:

op = "/mbz/users/liyuan/LLaMA-Factory/data/orca.json"
with open(op, "r") as read_file:
    data_s = json.load(read_file)
len(data_s)

3423052

## Calculate domain weights of original dataset

In [7]:
domain_lengths

defaultdict(int,
            {'t0': 732793216,
             'cot': 32259458,
             'flan': 465633957,
             'niv': 77085996})

In [12]:
domain_lengths = defaultdict(int)

# Iterate through each entry and sum the lengths
for entry in data:
    domain = entry.get("id", "Unknown Domain")
    total_len = entry.get("output_len", 0) + entry.get("input_len", 0)
    domain_lengths[domain] += total_len

domain_totals = defaultdict(int)

for source, total_length in domain_lengths.items():
    domain_totals[source] += total_length

# 4. Calculate the grand total (all domains combined)
grand_total = sum(domain_totals.values())

# 5. Display the results (Domain, Total Tokens, Proportion)
print("### Total Tokens per Domain\n")
print("| **Domain**         | **Total Tokens** | **Proportion (%)** |")
print("|--------------------|-------------------|---------------------|")

for domain, total in domain_totals.items():
    # Avoid division by zero (e.g., if grand_total = 0)
    if grand_total == 0:
        proportion = 0
    else:
        proportion = (total / grand_total) * 100

    # Format the domain label (replace underscores, capitalize)
    domain_label = domain
    print(f"| **{domain_label}** | {total:,} | {proportion:0.2f}% |")
    print("grand_total tokens:", grand_total)
    


### Total Tokens per Domain

| **Domain**         | **Total Tokens** | **Proportion (%)** |
|--------------------|-------------------|---------------------|
| **t0** | 732,793,216 | 56.03% |
grand_total tokens: 1307772627
| **cot** | 32,259,458 | 2.47% |
grand_total tokens: 1307772627
| **flan** | 465,633,957 | 35.61% |
grand_total tokens: 1307772627
| **niv** | 77,085,996 | 5.89% |
grand_total tokens: 1307772627


In [9]:
grand_total

1307772627

In [33]:
print(300_000_000 * 0.25)

75000000.0


## Sample validation dataset

In [ ]:
import json
import random
from collections import defaultdict

base_path = "/mbz/users/liyuan/LLaMA-Factory"
data_path = "/mbz/users/liyuan/LLaMA-Factory/data/orca_copy.json" # Adjust these paths as needed

with open(data_path, "r") as f:
    data = json.load(f)

# (Optional) Ensure each item has a truly unique ID:
import uuid
for item in data:
    if "unique_id" not in item:
        item["unique_id"] = str(uuid.uuid4())

# Build the sources dictionary by domain:
sources = defaultdict(list)
for item in data:
    domain = item["id"]  # or item["domain"] if your data uses that key
    sources[domain].append(item)

val_output_tokens_per_domain = 150000  # Rough target per domain
validation_data = defaultdict(list)
validation_items = []
source_list = ["t0", "cot", "flan", "niv"]

for domain in source_list:
    random.shuffle(sources[domain])
    
    domain_val_items = []
    current_tokens = 0
    
    # 1) First pass selection
    for item in sources[domain]:
        if current_tokens >= val_output_tokens_per_domain:
            break
        domain_val_items.append(item)
        current_tokens += item["output_len"]
    
    # 2) If still under target, collect from remaining
    if current_tokens < val_output_tokens_per_domain:
        remaining_items = [
            itm for itm in sources[domain]
            if itm not in domain_val_items
        ]
        random.shuffle(remaining_items)
        
        for item in remaining_items:
            if current_tokens >= val_output_tokens_per_domain:
                break
            domain_val_items.append(item)
            current_tokens += item["output_len"]
    
    # Store validation data for this domain
    validation_data[domain] = domain_val_items
    validation_items.extend(domain_val_items)
    
    # >>> Use a set of unique IDs to remove from sources <<<
    val_ids = {itm["unique_id"] for itm in domain_val_items}
    sources[domain] = [itm for itm in sources[domain] if itm["unique_id"] not in val_ids]
    
    print(f"Domain '{domain}': Selected {len(domain_val_items)} items "
          f"with total output tokens {current_tokens}")

print("\nValidation splits created for each domain.")
print(f"Total validation items across all domains: {len(validation_items)}")

# Save splits per domain

for domain in source_list:
    for item in validation_data[domain]:
        del item["input_len"]
        del item["output_len"]
        del item["unique_id"]
        del item["id"]
    
for domain in source_list:
    filename = f"{base_path}/data/data_mixing/orca/orca_{domain}_val.json"
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(validation_data[domain], f, indent=2)
    print(f"Saved validation data for domain '{domain}' to '{filename}'.")

# Save leftovers to train file
training_list = []
for domain in source_list:
    training_list.extend(sources[domain])

with open(f"{base_path}/data/data_mixing/orca/orca_sample.json", "w", encoding="utf-8") as f:
    json.dump(training_list, f, indent=2)

print("Remaining (training) data saved.")

Domain 't0': Selected 1407 items with total output tokens 150215
Domain 'cot': Selected 1022 items with total output tokens 150086
Domain 'flan': Selected 1164 items with total output tokens 150171
Domain 'niv': Selected 1010 items with total output tokens 150058

Validation splits created for each domain.
Total validation items across all domains: 4603
Saved validation data for domain 't0' to '/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/orca/orca_t0_val.json'.
Saved validation data for domain 'cot' to '/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/orca/orca_cot_val.json'.
Saved validation data for domain 'flan' to '/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/orca/orca_flan_val.json'.
Saved validation data for domain 'niv' to '/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/orca/orca_niv_val.json'.
Remaining (training) data saved.


In [5]:
with open(f"{base_path}/data/data_mixing/orca/orca_sample.json", "r") as f:
    data = json.load(f)
len(data)

3418449

In [29]:
import json
import random
import os
import copy

def load_data(file_path):
    """
    Load JSON data from a file.

    Parameters:
    - file_path (str): Path to the JSON file.

    Returns:
    - data (list of dict): The loaded JSON data.

    Raises:
    - FileNotFoundError: If the file does not exist.
    - json.JSONDecodeError: If the file contains invalid JSON.
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")
    
    with open(file_path, "r") as read_file:
        data = json.load(read_file)
    
    return data

def sample_tokens(data, limit, seed=42, max_passes=10):
    """
    Shuffle the data multiple times (upsampling) and sample items until the cumulative tokens
    (input_len + output_len) meets or exceeds the 'limit'. If all data is used and the limit
    is not yet met, reshuffle and continue sampling until 'max_passes' is reached.

    Once an item causes the cumulative tokens to exceed the limit, include that item and stop.
    After sampling, remove the 'output_len', 'input_len', and 'domain' keys from the items.

    Parameters:
    - data (list of dict): The input data to sample from.
    - limit (int): The token limit.
    - seed (int): The base random seed for shuffling.
    - max_passes (int): Maximum number of times to pass through the data for upsampling.

    Returns:
    - list of dict: The sampled items with specified keys removed.
    """
    total_tokens = 0
    sampled_items = []
    keys_to_remove = ["output_len", "input_len", "id", "unique_ids"]
    pass_num = 0  # To track the number of times we've looped through the data

    while total_tokens < limit and pass_num < max_passes:
        # Update the seed for each pass to ensure different shuffles
        current_seed = seed + pass_num
        random.seed(current_seed)
        
        shuffled_data = data.copy()
        random.shuffle(shuffled_data)
        
        for item in shuffled_data:
            item_tokens = item["output_len"] + item["input_len"]
            if total_tokens + item_tokens <= limit:
                sampled_items.append(item.copy())
                total_tokens += item_tokens
            else:
                # Adding this item would exceed the limit, include it and stop
                sampled_items.append(item.copy())
                total_tokens += item_tokens
                # After adding the item, exit all loops
                break
        else:
            # Completed a full pass without exceeding the limit
            pass_num += 1
            continue  # Continue to the next pass if the limit is not yet met
        break  # Exit the while loop after the break in for-loop

    if total_tokens < limit:
        print(f"Warning: Reached maximum passes ({max_passes}) without meeting token limit.")

    # Remove specified keys from sampled items
    for item in sampled_items:
        del item["output_len"]
        del item["input_len"]
        del item["id"]
        del item["unique_id"]
    
    return sampled_items

def ensure_directory_exists(directory_path):
    """
    Ensure that the specified directory exists. If not, create it.

    Parameters:
    - directory_path (str): Path to the directory.
    """
    os.makedirs(directory_path, exist_ok=True)

def main():
    # Define file paths and parameters
    data_file_path = f"/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/{experiment_name}/{experiment_name}_sample.json"
    output_dir = f"/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/{experiment_name}"
    
    # Load data
    try:
        data = load_data(data_file_path)
    except (FileNotFoundError, json.JSONDecodeError) as e:
        print(f"Error loading data: {e}")
        return
    
    # Define token limits
    token_limits = {
        "t0":   168090000,
        "cot":  7410000,
        "flan": 106830000,
        "niv":  17670000,
    }
    
    # Define domains to process
    domains = ["t0", "cot", "flan", "niv"]
    total_data = []
    # Ensure the output directory exists
    ensure_directory_exists(output_dir)
    
    for domain in domains:
        # Filter the data by domain
        domain_data = [item for item in data if item["id"] == domain]
        
        if not domain_data:
            print(f"No data found for domain: {domain}. Skipping.")
            continue
        
        limit = token_limits[domain]
        # Sample tokens with upsampling
        sampled_items = sample_tokens(domain_data, limit)
        total_data.extend(sampled_items)
        
        # Print stats
        print(f"Domain: {domain} | Limit: {limit:,} | Items: {len(sampled_items)}")
        
    with open(f"/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/{experiment_name}/{experiment_name}_original.json", "w") as wri_f:
        json.dump(total_data, wri_f, indent = 4)

if __name__ == "__main__":
    main()

Domain: t0 | Limit: 168,090,000 | Items: 381417
Domain: cot | Limit: 7,410,000 | Items: 32053
Domain: flan | Limit: 106,830,000 | Items: 324354
Domain: niv | Limit: 17,670,000 | Items: 47421


In [30]:
with open(f"/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/{experiment_name}/{experiment_name}_original.json", "r") as wri_f:
    d = json.load(wri_f)
print(len(d))

785245


## Sample equal for orca

In [34]:
def run_orca_equal():
    # Define file paths and parameters
    data_file_path = f"/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/{experiment_name}/{experiment_name}_sample.json"
    output_dir = f"/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/{experiment_name}"
    
    # Load data
    try:
        data = load_data(data_file_path)
    except (FileNotFoundError, json.JSONDecodeError) as e:
        print(f"Error loading data: {e}")
        return
    
    # Define token limits
    token_limits = {
        "t0":   75000000,
        "cot":  75000000,
        "flan": 75000000,
        "niv":  75000000,
    }
    
    # Define domains to process
    domains = ["t0", "cot", "flan", "niv"]
    total_data = []
    # Ensure the output directory exists
    ensure_directory_exists(output_dir)
    
    for domain in domains:
        # Filter the data by domain
        domain_data = [item for item in data if item["id"] == domain]
        
        if not domain_data:
            print(f"No data found for domain: {domain}. Skipping.")
            continue
        
        limit = token_limits[domain]
        # Sample tokens with upsampling
        sampled_items = sample_tokens(domain_data, limit)
        total_data.extend(sampled_items)
        
        # Print stats
        print(f"Domain: {domain} | Limit: {limit:,} | Items: {len(sampled_items)}")
        
    with open(f"/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/{experiment_name}/{experiment_name}_equal.json", "w") as wri_f:
        json.dump(total_data, wri_f, indent = 4)

run_orca_equal()

Domain: t0 | Limit: 75,000,000 | Items: 170899
Domain: cot | Limit: 75,000,000 | Items: 325228
Domain: flan | Limit: 75,000,000 | Items: 228179
Domain: niv | Limit: 75,000,000 | Items: 201489


In [35]:
with open(f"/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/{experiment_name}/{experiment_name}_equal.json", "r") as wri_f:
    d = json.load(wri_f)
print(len(d))

925795


In [ ]:
dataset_info_path = "/mbz/users/liyuan/LLaMA-Factory/data/dataset_info.json"
with open(dataset_info_path, "r") as f:
    try:
        dataset_info = json.load(f)
    except json.JSONDecodeError:
        dataset_info = {}
        
# domains = ["code", "general", "knowledge_recall", "math", "precise_IF", "safety"]
# token_limits = {
#         "equal": int(462841566/6),
#     }


dataset_name = f"orca_equal"
output_path = f"data_mixing/{experiment_name}/{dataset_name}.json"
dataset_info[dataset_name] = {
    "file_name": output_path
}
    
with open(dataset_info_path, "w") as f:
    json.dump(dataset_info, f, indent=2)

## Sample 660K

In [9]:
import json
import random
import os
import copy

def load_data(file_path):
    """
    Load JSON data from a file.

    Parameters:
    - file_path (str): Path to the JSON file.

    Returns:
    - data (list of dict): The loaded JSON data.

    Raises:
    - FileNotFoundError: If the file does not exist.
    - json.JSONDecodeError: If the file contains invalid JSON.
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")
    
    with open(file_path, "r") as read_file:
        data = json.load(read_file)
    
    return data

def sample_tokens(data, limit, seed=42, max_passes=10):
    """
    Shuffle the data multiple times (upsampling) and sample items until the cumulative tokens
    (input_len + output_len) meets or exceeds the 'limit'. If all data is used and the limit
    is not yet met, reshuffle and continue sampling until 'max_passes' is reached.

    Once an item causes the cumulative tokens to exceed the limit, include that item and stop.
    After sampling, remove the 'output_len', 'input_len', and 'domain' keys from the items.

    Parameters:
    - data (list of dict): The input data to sample from.
    - limit (int): The token limit.
    - seed (int): The base random seed for shuffling.
    - max_passes (int): Maximum number of times to pass through the data for upsampling.

    Returns:
    - list of dict: The sampled items with specified keys removed.
    """
    total_tokens = 0
    sampled_items = []
    keys_to_remove = ["output_len", "input_len", "id", "unique_id"]
    pass_num = 0  # To track the number of times we've looped through the data

    while total_tokens < limit and pass_num < max_passes:
        # Update the seed for each pass to ensure different shuffles
        current_seed = seed + pass_num
        random.seed(current_seed)
        
        shuffled_data = data.copy()
        random.shuffle(shuffled_data)
        
        for item in shuffled_data:
            item_tokens = item["output_len"] + item["input_len"]
            if total_tokens + item_tokens <= limit:
                sampled_items.append(item.copy())
                total_tokens += item_tokens
            else:
                # Adding this item would exceed the limit, include it and stop
                sampled_items.append(item.copy())
                total_tokens += item_tokens
                # After adding the item, exit all loops
                break
        else:
            # Completed a full pass without exceeding the limit
            pass_num += 1
            continue  # Continue to the next pass if the limit is not yet met
        break  # Exit the while loop after the break in for-loop

    if total_tokens < limit:
        print(f"Warning: Reached maximum passes ({max_passes}) without meeting token limit.")

    # Remove specified keys from sampled items
    for item in sampled_items:
        del item["output_len"]
        del item["input_len"]
        del item["id"]
        del item["unique_id"]
    
    return sampled_items

def ensure_directory_exists(directory_path):
    """
    Ensure that the specified directory exists. If not, create it.

    Parameters:
    - directory_path (str): Path to the directory.
    """
    os.makedirs(directory_path, exist_ok=True)

def main():
    # Define file paths and parameters
    data_file_path = "/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/orca/orca_sample.json"
    output_dir = "/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/orca"
    base_token = 660_000
    
    # Load data
    try:
        data = load_data(data_file_path)
    except (FileNotFoundError, json.JSONDecodeError) as e:
        print(f"Error loading data: {e}")
        return
    
    # Define token limits
    token_limits = {
        "one": base_token,
        "half": base_token // 2,
        "third": base_token // 3,
        "double": base_token * 2,
        "triple": base_token * 3
    }
    
    # Define domains to process
    domains = ["t0", "cot", "flan", "niv"]
    
    # Ensure the output directory exists
    ensure_directory_exists(output_dir)
    
    for domain in domains:
        # Filter the data by domain
        domain_data = [item for item in data if item["id"] == domain]
        
        if not domain_data:
            print(f"No data found for domain: {domain}. Skipping.")
            continue
        
        for name, limit in token_limits.items():
            # Sample tokens with upsampling
            sampled_items = sample_tokens(domain_data, limit)
            
            # Define output file path
            out_filename = f"{base_token}_{domain}_{name}.json"
            out_path = os.path.join(output_dir, out_filename)
            
            # Save sampled items to JSON
            try:
                with open(out_path, "w") as f:
                    json.dump(sampled_items, f, indent=4)
            except IOError as e:
                print(f"Error writing to file {out_path}: {e}")
                continue
            
            # Print stats
            print(f"Domain: {domain} | Limit: {limit:,} | Items: {len(sampled_items)}")
    
if __name__ == "__main__":
    main()

Domain: t0 | Limit: 660,000 | Items: 1540
Domain: t0 | Limit: 330,000 | Items: 751
Domain: t0 | Limit: 220,000 | Items: 507
Domain: t0 | Limit: 1,320,000 | Items: 3039
Domain: t0 | Limit: 1,980,000 | Items: 4563
Domain: cot | Limit: 660,000 | Items: 2870
Domain: cot | Limit: 330,000 | Items: 1447
Domain: cot | Limit: 220,000 | Items: 960
Domain: cot | Limit: 1,320,000 | Items: 5740
Domain: cot | Limit: 1,980,000 | Items: 8600
Domain: flan | Limit: 660,000 | Items: 1975
Domain: flan | Limit: 330,000 | Items: 1020
Domain: flan | Limit: 220,000 | Items: 683
Domain: flan | Limit: 1,320,000 | Items: 4003
Domain: flan | Limit: 1,980,000 | Items: 6048
Domain: niv | Limit: 660,000 | Items: 1778
Domain: niv | Limit: 330,000 | Items: 907
Domain: niv | Limit: 220,000 | Items: 584
Domain: niv | Limit: 1,320,000 | Items: 3535
Domain: niv | Limit: 1,980,000 | Items: 5309


In [11]:
dataset_info_path = "/mbz/users/liyuan/LLaMA-Factory/data/dataset_info.json"
with open(dataset_info_path, "r") as f:
    try:
        dataset_info = json.load(f)
    except json.JSONDecodeError:
        dataset_info = {}
        
domains = ["t0", "cot", "flan", "niv"]
base_token = 660_000
token_limits = {
        "one": base_token,
        "half": base_token // 2,
        "third": base_token // 3,
        "double": base_token * 2,
        "triple": base_token * 3
    }
experiment_name="orca"
for domain in domains:
    # dataset_name = f"{base_token}_{domain}_val"
    # output_path = f"data_mixing/{experiment_name}/{dataset_name}.json"
    # dataset_info[dataset_name] = {
    #     "file_name": output_path
    # }
    
    for size in token_limits.keys():
        dataset_name = f"{base_token}_{domain}_{size}"
        output_path = f"data_mixing/{experiment_name}/{dataset_name}.json"
        dataset_info[dataset_name] = {
            "file_name": output_path
        }
        
    with open(dataset_info_path, "w") as f:
        json.dump(dataset_info, f, indent=2)

In [ ]:
dataset_info_path = "/mbz/users/liyuan/LLaMA-Factory/data/dataset_info.json"
with open(dataset_info_path, "r") as f:
    try:
        dataset_info = json.load(f)
    except json.JSONDecodeError:
        dataset_info = {}

    # for size in ["one", "half", "third", "double", "triple"]:
    dataset_name = f"tulu3_original"
    output_path = f"/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/tulu3/tulu3_original.json"
    dataset_info[dataset_name] = {
        "file_name": output_path
    }
    
    for domain in ["t0", "cot", "flan", "niv"]:
        dataset_name = f"orca_{domain}_val"
        output_path = f"data_mixing/orca/{dataset_name}.json"
        dataset_info[dataset_name] = {
            "file_name": output_path
        }
    # # for size in ["one", "half", "third", "double", "triple"]:
    # for size in ["onehalf"]:
    #     dataset_name = f"{domain}_{size}"
    #     output_path = f"data_mixing/{base_token}_{dataset_name}.json"
    #     dataset_info[dataset_name] = {
    #         "file_name": output_path
    #     }
        
with open(dataset_info_path, "w") as f:
    json.dump(dataset_info, f, indent=2)

## Equal weights for Tulu3

In [4]:

domains = ["code", "general", "knowledge_recall", "math", "precise_IF", "safety"]
datafile = [f"/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/tulu3/{domain}_equal.json" for domain in domains]

combined_data = []
for data_source in datafile:
    with open(data_source, "r") as read_file:
        data = json.load(read_file)
    combined_data.extend(data)
    
dataset_name = f"tulu3_equal"
with open(f"/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/tulu3/{dataset_name}.json", "w") as f:
    json.dump(combined_data, f, indent=2)

In [11]:
dataset_info_path = "/mbz/users/liyuan/LLaMA-Factory/data/dataset_info.json"
with open(dataset_info_path, "r") as f:
    try:
        dataset_info = json.load(f)
    except json.JSONDecodeError:
        dataset_info = {}
        
domains = ["code", "general", "knowledge_recall", "math", "precise_IF", "safety"]
token_limits = {
        "equal": int(462841566/6),
    }
experiment_name="tulu3"
for domain in domains:
    
    for size in token_limits.keys():
        dataset_name = f"{domain}_{size}"
        output_path = f"data_mixing/{experiment_name}/{dataset_name}.json"
        dataset_info[dataset_name] = {
            "file_name": output_path
        }
    
    dataset_name = f"tulu3_equal"
    output_path = f"data_mixing/{experiment_name}/{dataset_name}.json"
    dataset_info[dataset_name] = {
        "file_name": output_path
    }
        
    with open(dataset_info_path, "w") as f:
        json.dump(dataset_info, f, indent=2)

In [13]:
len(data)

1092938

In [2]:
import json
import random
import os

# Optional: Set a seed for reproducibility
random.seed(42)

# Define the path to your input and output files
input_file = "/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/orca/orca_sample.json"
output_file = "/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/orca/orca_sampled.json"  # Update as needed

# Number of samples per 'id' type
SAMPLES_PER_ID = 3000

# Ensure the input file exists
if not os.path.exists(input_file):
    raise FileNotFoundError(f"Input file not found: {input_file}")

# Load the data from the JSON file
with open(input_file, "r", encoding="utf-8") as f:
    try:
        data = json.load(f)
    except json.JSONDecodeError as e:
        raise ValueError(f"Error decoding JSON: {e}")

# Verify that the data is a list
if not isinstance(data, list):
    raise TypeError("JSON data should be a list of dictionaries.")

# Group data by 'id'
grouped_data = {"t0": [], "cot": [], "flan": [], "niv": []}

for entry in data:
    entry_id = entry.get('id')
    if entry_id in grouped_data:
        grouped_data[entry_id].append(entry)
    else:
        # If there are other 'id' types, you can choose to handle them or ignore
        pass  # Ignoring other 'id' types for this task

# Initialize a list to hold the sampled data
sampled_data = []

# Sample 1000 entries from each 'id' group
for id_type, entries in grouped_data.items():
    total_entries = len(entries)
    if total_entries < SAMPLES_PER_ID:
        print(f"Warning: '{id_type}' has only {total_entries} entries. Sampling all available entries.")
        sampled_entries = entries.copy()
    else:
        sampled_entries = random.sample(entries, SAMPLES_PER_ID)
    sampled_data.extend(sampled_entries)

# Optional: Shuffle the sampled data to mix different 'id' types
random.shuffle(sampled_data)

# Save the sampled data to the output JSON file
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(sampled_data, f, ensure_ascii=False, indent=2)

print(f"Sampled data has been saved to {output_file}")


Sampled data has been saved to /mbz/users/liyuan/LLaMA-Factory/data/data_mixing/orca/orca_sampled.json


In [55]:
import json
import random
import os

# Optional: Set a seed for reproducibility
random.seed(42)

# Define the path to your input and output files
input_file = "/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/orca/orca_sample.json"
output_file = "/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/orca/orca_submodular.json"  # Update as needed

# Number of samples per 'id' type
SAMPLES_PER_ID = 248090

# Ensure the input file exists
if not os.path.exists(input_file):
    raise FileNotFoundError(f"Input file not found: {input_file}")

# Load the data from the JSON file
with open(input_file, "r", encoding="utf-8") as f:
    try:
        data = json.load(f)
    except json.JSONDecodeError as e:
        raise ValueError(f"Error decoding JSON: {e}")

# Verify that the data is a list
if not isinstance(data, list):
    raise TypeError("JSON data should be a list of dictionaries.")

# Group data by 'id'
grouped_data = {"t0": [], "cot": [], "flan": [], "niv": []}

for entry in data:
    entry_id = entry.get('id')
    if entry_id in grouped_data:
        grouped_data[entry_id].append(entry)
    else:
        # If there are other 'id' types, you can choose to handle them or ignore
        pass  # Ignoring other 'id' types for this task

# Initialize a list to hold the sampled data
sampled_data = []

# Sample entries from each 'id' group
for id_type, entries in grouped_data.items():
    total_entries = len(entries)
    if total_entries == 0:
        print(f"Warning: '{id_type}' has no entries. Skipping this group.")
        continue  # Skip this group as there are no entries to sample from

    if total_entries < SAMPLES_PER_ID:
        print(f"Warning: '{id_type}' has only {total_entries} entries. Resampling with replacement to reach {SAMPLES_PER_ID}.")
        # Use random.choices to sample with replacement
        sampled_entries = random.choices(entries, k=SAMPLES_PER_ID)
    else:
        # Sample without replacement
        sampled_entries = random.sample(entries, SAMPLES_PER_ID)
    
    sampled_data.extend(sampled_entries)

# Optional: Shuffle the sampled data to mix different 'id' types
random.shuffle(sampled_data)

# Save the sampled data to the output JSON file
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(sampled_data, f, ensure_ascii=False, indent=2)

print(f"Sampled data has been saved to {output_file}")


Sampled data has been saved to /mbz/users/liyuan/LLaMA-Factory/data/data_mixing/orca/orca_submodular.json


In [53]:
grouped_data = {}
for item in data:
    domain_id = item["id"]
    if domain_id not in grouped_data:
        grouped_data[domain_id] = []
    grouped_data[domain_id].append(item)

# -----------------------------
# 3) (Optional) Shuffle each domain
#    so when we slice the first S, it's effectively a random S-sample
# -----------------------------
for domain_id, items in grouped_data.items():
    random.shuffle(items)

# -----------------------------
# 4) Repeatedly prompt the user for S
#    and compute the total tokens
# -----------------------------

S = 248090

total_tokens = 0
# For each domain, pick the first S items
for domain_id, items in grouped_data.items():
    selected = items[:S]  # if S > len(items), this will just be all items
    for e in selected:
        total_tokens += (e["input_len"] + e["output_len"])

print(f"Total tokens for S = {S} is {total_tokens}\n")

Total tokens for S = 248090 is 300045665



In [74]:
import json
import random
import os

# Optional: Set a seed for reproducibility
random.seed(42)

# Define the path to your input and output files
input_file = "/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/orca/orca_sample.json"
output_file = "/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/orca/orca_submodular.json"  # Update as needed

# Number of samples per 'id' type
SAMPLES_PER_ID = 222000

# Ensure the input file exists
if not os.path.exists(input_file):
    raise FileNotFoundError(f"Input file not found: {input_file}")

# Load the data from the JSON file
with open(input_file, "r", encoding="utf-8") as f:
    try:
        data = json.load(f)
    except json.JSONDecodeError as e:
        raise ValueError(f"Error decoding JSON: {e}")

# Verify that the data is a list
if not isinstance(data, list):
    raise TypeError("JSON data should be a list of dictionaries.")

# Group data by 'id'
grouped_data = {"t0": [], "cot": [], "flan": [], "niv": []}

for entry in data:
    entry_id = entry.get('id')
    if entry_id in grouped_data:
        grouped_data[entry_id].append(entry)
    else:
        # If there are other 'id' types, you can choose to handle them or ignore
        pass  # Ignoring other 'id' types for this task

# Initialize a list to hold the sampled data
sampled_data = []

# Sample entries from each 'id' group
for id_type, entries in grouped_data.items():
    total_entries = len(entries)
    if total_entries == 0:
        print(f"Warning: '{id_type}' has no entries. Skipping this group.")
        continue  # Skip this group as there are no entries to sample from

    if total_entries < SAMPLES_PER_ID:
        print(f"Warning: '{id_type}' has only {total_entries} entries. Resampling with replacement to reach {SAMPLES_PER_ID}.")
        # Use random.choices to sample with replacement
        sampled_entries = random.choices(entries, k=SAMPLES_PER_ID)
    else:
        # Sample without replacement
        sampled_entries = random.sample(entries, SAMPLES_PER_ID)
    
    sampled_data.extend(sampled_entries)

# Optional: Shuffle the sampled data to mix different 'id' types
random.shuffle(sampled_data)

# Save the sampled data to the output JSON file
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(sampled_data, f, ensure_ascii=False, indent=2)

print(f"Sampled data has been saved to {output_file}")

Sampled data has been saved to /mbz/users/liyuan/LLaMA-Factory/data/data_mixing/orca/orca_submodular.json


In [17]:
import json
with open("/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/orca/orca_submodular.json", "r") as file:
    data = json.load(file)
    
len(data)
data[0]

{'instruction': 'You are a teacher. Given a task, you explain in simple steps what the task is asking, any guidelines it provides and how to use those guidelines to find the answer.\nDefinition: This task is about using the specified sentence and converting the sentence to Resource Description Framework (RDF) triplets of the form (subject, predicate object). The RDF triplets generated must be such that the triplets accurately capture the structure and semantics of the input sentence. The input is a sentence and the output is a list of triplets of the form [subject, predicate, object] that capture the relationships present in the sentence. When a sentence has more than 1 RDF triplet possible, the output must contain all of them.\nInput: A moderately priced and rated Indian restaurant in the Riverside area would be The Phoenix.\nOutput:',
 'input': '',
 'output': 'The task is asking you to take the given sentence and convert it into a list of RDF triplets. RDF triplets are made up of thr

In [16]:
f_path = "/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/orca/orca_submodular.json"
structure_data = []
for item in data:
    del item["id"]
    del item["input_len"]
    del item["output_len"]
    del item["unique_id"]
    structure_data.append(item)

with open(f_path, "w") as file:
    json.dump(structure_data,file, indent=4)


## Our method

In [6]:
import json
import random
import os
import copy

def load_data(file_path):
    """
    Load JSON data from a file.

    Parameters:
    - file_path (str): Path to the JSON file.

    Returns:
    - data (list of dict): The loaded JSON data.

    Raises:
    - FileNotFoundError: If the file does not exist.
    - json.JSONDecodeError: If the file contains invalid JSON.
    """
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")
    
    with open(file_path, "r") as read_file:
        data = json.load(read_file)
    
    return data

def sample_tokens(data, limit, seed=42, max_passes=10):
    """
    Shuffle the data multiple times (upsampling) and sample items until the cumulative tokens
    (input_len + output_len) meets or exceeds the 'limit'. If all data is used and the limit
    is not yet met, reshuffle and continue sampling until 'max_passes' is reached.

    Once an item causes the cumulative tokens to exceed the limit, include that item and stop.
    After sampling, remove the 'output_len', 'input_len', and 'domain' keys from the items.

    Parameters:
    - data (list of dict): The input data to sample from.
    - limit (int): The token limit.
    - seed (int): The base random seed for shuffling.
    - max_passes (int): Maximum number of times to pass through the data for upsampling.

    Returns:
    - list of dict: The sampled items with specified keys removed.
    """
    total_tokens = 0
    sampled_items = []
    keys_to_remove = ["output_len", "input_len", "id", "unique_ids"]
    pass_num = 0  # To track the number of times we've looped through the data

    while total_tokens < limit and pass_num < max_passes:
        # Update the seed for each pass to ensure different shuffles
        current_seed = seed + pass_num
        random.seed(current_seed)
        
        shuffled_data = data.copy()
        random.shuffle(shuffled_data)
        
        for item in shuffled_data:
            item_tokens = item["output_len"] + item["input_len"]
            if total_tokens + item_tokens <= limit:
                sampled_items.append(item.copy())
                total_tokens += item_tokens
            else:
                # Adding this item would exceed the limit, include it and stop
                sampled_items.append(item.copy())
                total_tokens += item_tokens
                # After adding the item, exit all loops
                break
        else:
            # Completed a full pass without exceeding the limit
            pass_num += 1
            continue  # Continue to the next pass if the limit is not yet met
        break  # Exit the while loop after the break in for-loop

    if total_tokens < limit:
        print(f"Warning: Reached maximum passes ({max_passes}) without meeting token limit.")

    # Remove specified keys from sampled items
    for item in sampled_items:
        del item["output_len"]
        del item["input_len"]
        del item["id"]
        del item["unique_id"]
    
    return sampled_items

def ensure_directory_exists(directory_path):
    """
    Ensure that the specified directory exists. If not, create it.

    Parameters:
    - directory_path (str): Path to the directory.
    """
    os.makedirs(directory_path, exist_ok=True)

def main():
    # Define file paths and parameters
    data_file_path = f"/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/{experiment_name}/{experiment_name}_sample.json"
    output_dir = f"/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/{experiment_name}"
    
    # Load data
    try:
        data = load_data(data_file_path)
    except (FileNotFoundError, json.JSONDecodeError) as e:
        print(f"Error loading data: {e}")
        return
    
    # Define token limits
    rate = [0.35983086, 0.20161636, 0.34249649, 0.096]
    token_limits = {
        "t0":   int(300_000_000 * rate[0]),
        "cot":  int(300_000_000 * rate[1]),
        "flan": int(300_000_000 * rate[2]),
        "niv":  int(300_000_000 * rate[3]),
    }
    
    # Define domains to process
    domains = ["t0", "cot", "flan", "niv"]
    total_data = []
    # Ensure the output directory exists
    ensure_directory_exists(output_dir)
    
    for domain in domains:
        # Filter the data by domain
        domain_data = [item for item in data if item["id"] == domain]
        
        if not domain_data:
            print(f"No data found for domain: {domain}. Skipping.")
            continue
        
        limit = token_limits[domain]
        # Sample tokens with upsampling
        sampled_items = sample_tokens(domain_data, limit)
        total_data.extend(sampled_items)
        
        # Print stats
        print(f"Domain: {domain} | Limit: {limit:,} | Items: {len(sampled_items)}")
        
    with open(f"/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/{experiment_name}/{experiment_name}_Qwen_ours.json", "w") as wri_f:
        json.dump(total_data, wri_f, indent = 4)

if __name__ == "__main__":
    main()



Domain: t0 | Limit: 107,949,257 | Items: 245344
Domain: cot | Limit: 60,484,908 | Items: 262431
Domain: flan | Limit: 102,748,947 | Items: 311968
Domain: niv | Limit: 28,800,000 | Items: 77227


In [7]:
with open(f"/mbz/users/liyuan/LLaMA-Factory/data/data_mixing/{experiment_name}/{experiment_name}_ours.json", "r") as read_f:
    data = json.load(read_f)
    
print(len(data))


1097727


In [7]:
dataset_info_path = "/mbz/users/liyuan/LLaMA-Factory/data/dataset_info.json"
with open(dataset_info_path, "r") as f:
    try:
        dataset_info = json.load(f)
    except json.JSONDecodeError:
        dataset_info = {}
        
experiment_name="orca"

dataset_name = f"orca_Qwen_ours"
output_path = f"data_mixing/{experiment_name}/{dataset_name}.json"
dataset_info[dataset_name] = {
    "file_name": output_path
}
    
with open(dataset_info_path, "w") as f:
    json.dump(dataset_info, f, indent=2)